In [ ]:
# data_ingest_sqlite.py

import sqlite3
import pandas as pd
from pathlib import Path

CSV_PATH = Path("resqfood_listings.csv")
DB_PATH = Path("resqfood.db")

# ---------------------------------
# FILE VALIDATION
# ---------------------------------
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV file not found: {CSV_PATH}")

print("Loading CSV file...")

df = pd.read_csv(CSV_PATH)

# ---------------------------------
# REQUIRED COLUMNS CHECK
# ---------------------------------
required_cols = [
    "listing_id",
    "donor_id",
    "donor_type",
    "food_type",
    "quantity",
    "unit",
    "city",
    "packaging",
    "time_posted",
    "time_since_cooked_mins",
    "donor_reliability",
    "distance_km",
    "ambient_temp_c",
    "freshness_score",
    "pickup_prob",
    "picked_up",
]

missing_cols = [
    col for col in required_cols
    if col not in df.columns
]

if missing_cols:
    raise ValueError(
        f"Missing columns: {missing_cols}"
    )

# ---------------------------------
# REMOVE DUPLICATES
# ---------------------------------
df = df.drop_duplicates(
    subset=["listing_id"]
)

# ---------------------------------
# DATETIME CLEANING
# ---------------------------------
df["time_posted"] = pd.to_datetime(
    df["time_posted"],
    errors="coerce"
)

df = df.dropna(
    subset=["time_posted"]
)

df["time_posted"] = (
    df["time_posted"]
    .dt.strftime("%Y-%m-%d %H:%M:%S")
)

# ---------------------------------
# NUMERIC CONVERSIONS
# ---------------------------------
numeric_cols = [
    "quantity",
    "time_since_cooked_mins",
    "distance_km",
    "ambient_temp_c",
    "freshness_score",
    "pickup_prob",
    "picked_up",
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# ---------------------------------
# HANDLE MISSING VALUES
# ---------------------------------
for col in [
    "distance_km",
    "ambient_temp_c",
    "freshness_score",
    "pickup_prob",
]:
    df[col] = df[col].fillna(
        df[col].median()
    )

df["quantity"] = df["quantity"].fillna(0)

df["time_since_cooked_mins"] = (
    df["time_since_cooked_mins"]
    .fillna(0)
    .astype(int)
)

df["picked_up"] = (
    df["picked_up"]
    .fillna(0)
    .astype(int)
)

# ---------------------------------
# EMPTY DATA CHECK
# ---------------------------------
if len(df) == 0:
    raise ValueError(
        "No valid rows remaining after cleaning."
    )

print(
    f"Final rows after cleaning: {len(df)}"
)

# ---------------------------------
# SQLITE DATABASE
# ---------------------------------
with sqlite3.connect(DB_PATH) as conn:

    cursor = conn.cursor()

    cursor.execute("""
    CREATE TABLE IF NOT EXISTS listings (
        listing_id TEXT PRIMARY KEY,
        donor_id TEXT,
        donor_type TEXT,
        food_type TEXT,
        quantity REAL,
        unit TEXT,
        city TEXT,
        packaging TEXT,
        time_posted TEXT,
        time_since_cooked_mins INTEGER,
        donor_reliability TEXT,
        distance_km REAL,
        ambient_temp_c REAL,
        freshness_score REAL,
        pickup_prob REAL,
        picked_up INTEGER
    )
    """)

    # Remove old data
    cursor.execute(
        "DELETE FROM listings"
    )

    conn.commit()

    # Insert cleaned data
    df.to_sql(
        "listings",
        conn,
        if_exists="append",
        index=False
    )

    # ---------------------------------
    # INDEXES
    # ---------------------------------
    cursor.execute("""
    CREATE INDEX IF NOT EXISTS
    idx_time_posted
    ON listings(time_posted)
    """)

    cursor.execute("""
    CREATE INDEX IF NOT EXISTS
    idx_city
    ON listings(city)
    """)

    cursor.execute("""
    CREATE INDEX IF NOT EXISTS
    idx_pickup
    ON listings(picked_up)
    """)

    conn.commit()

    # Verify records
    count = cursor.execute(
        "SELECT COUNT(*) FROM listings"
    ).fetchone()[0]

print(
    f" Data successfully ingested into {DB_PATH}"
)
print(
    f" Total records inserted: {count}"
)

Ingested into resqfood.db
